In [43]:
import torch
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [44]:
# 1. Setup paths and verify GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device acceleration: {device.upper()}")

processed_dir = Path.cwd().parent / 'dataset' / 'processed'
train_path = processed_dir / 'clf_train_labeled_v1.parquet'
val_path = processed_dir / 'clf_val_labeled_v1.parquet'

Using device acceleration: CUDA


In [45]:
df_train = pd.read_parquet(train_path).dropna(subset=['text', 'predicted_intent'])
df_val = pd.read_parquet(val_path).dropna(subset=['text', 'predicted_intent'])

# 2. Map categorical string intents to explicit numeric IDs
unique_intents = sorted(df_train['predicted_intent'].unique().tolist())
intent2id = {intent: idx for idx, intent in enumerate(unique_intents)}
id2intent = {idx: intent for intent, idx in intent2id.items()}

df_train['label'] = df_train['predicted_intent'].map(intent2id)
df_val['label'] = df_val['predicted_intent'].map(intent2id)

# 3. Convert Pandas DataFrames into optimized HuggingFace Dataset classes
train_dataset = Dataset.from_pandas(df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text', 'label']])

In [46]:
# 4. Initialize the DistilBERT Tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 1419/1419 [00:00<00:00, 11484.95 examples/s]


In [47]:
# 5. Initialize the Model Architecture
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=len(unique_intents),
    id2label=id2intent,
    label2id=intent2id
).to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 945.66it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [48]:
# 6. Define the Automated Evaluation Harness Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [49]:
# 7. Configure Training Hyperparameters (Optimized defensively for a 6GB VRAM footprint)
training_args = TrainingArguments(
    output_dir="./results_distilbert",
    eval_strategy="epoch",            # Evaluate at the end of every epoch
    save_strategy="epoch",            # Save checkpoints safely every epoch
    learning_rate=3e-5,               # Safe standard rate for sequence classification
    per_device_train_batch_size=16,   # Low batch size prevents 6GB VRAM OOM crashes
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # 3 passes over the dataset is standard for distillation
    weight_decay=0.01,                # Basic regularization to stop overfitting
    load_best_model_at_end=True,      # Keep the checkpoint with the highest validation metrics
    metric_for_best_model="f1",
    fp16=True,                        # Crucial! Uses mixed precision to double processing speed on NVIDIA cards
    logging_steps=100,
    report_to="none"                  # Turn off external wandb tracking noise
)

In [50]:
# 8. Start the Fine-Tuning Execution Engine
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("\n--- Fine-Tuning DistilBERT Student Model ---")
trainer.train()


--- Fine-Tuning DistilBERT Student Model ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.085487,0.958898,0.678647,0.665296,0.702556,0.678647
2,0.745304,0.861106,0.712474,0.715318,0.730822,0.712474
3,0.535863,0.828139,0.728682,0.727908,0.729961,0.728682


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.07it/s]


TrainOutput(global_step=2130, training_loss=0.9089625416787017, metrics={'train_runtime': 184.7779, 'train_samples_per_second': 184.259, 'train_steps_per_second': 11.527, 'total_flos': 1127770675140096.0, 'train_loss': 0.9089625416787017, 'epoch': 3.0})

In [51]:
# Create a dedicated local models folder inside your project tree
local_model_dir = Path.cwd().parent / "models" / "intent_classifier_distilbert"
local_model_dir.mkdir(parents=True, exist_ok=True)

# Freeze weights and vocabulary states to disk
trainer.save_model(local_model_dir)
tokenizer.save_pretrained(local_model_dir)

print(f"Success! Best model files safely isolated at: {local_model_dir}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]

Success! Best model files safely isolated at: D:\Projects\Customer-Support-Agent\models\intent_classifier_distilbert


In [15]:
# Calculate total steps: (Total samples / Batch Size) * Epochs
total_train_samples = len(tokenized_train)
batch_size = 16
epochs = 5

total_steps = (total_train_samples // batch_size) * epochs
# 10% of total steps for warmup
manual_warmup_steps = int(0.10 * total_steps)

print(f"Total Training Steps: {total_steps}")
print(f"10% Warmup Steps to inject: {manual_warmup_steps}")

Total Training Steps: 3545
10% Warmup Steps to inject: 354


In [16]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

tuned_training_args = TrainingArguments(
    output_dir="./results_distilbert_tuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,               # Marginally tighter learning rate for finer optimization
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,               # Expanded training headroom
    weight_decay=0.02,                # Increased regularisation to accommodate extra epochs
    warmup_steps=manual_warmup_steps, # FIXED: Replaced ratio with explicit numerical steps
    lr_scheduler_type="cosine",       # Advanced decay schedule
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
    report_to="none"
)

# Initialize the automated execution block with an early stopping guardrail
tuned_trainer = Trainer(
    model=model,                      
    args=tuned_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stops run if validation degrades
)

print("\n--- Running Optimized Hyperparameter Fine-Tuning Loop ---")
tuned_trainer.train()


--- Running Optimized Hyperparameter Fine-Tuning Loop ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.037281,1.880663,0.696970,0.696037,0.700977,0.696970
2,0.056033,2.103729,0.683580,0.687098,0.703350,0.683580
3,0.078348,2.006624,0.701903,0.702908,0.706144,0.701903
4,0.030344,2.097369,0.709655,0.710594,0.714092,0.709655
5,0.017924,2.105782,0.705426,0.705080,0.707365,0.705426


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]


TrainOutput(global_step=3550, training_loss=0.04314749269418314, metrics={'train_runtime': 310.4337, 'train_samples_per_second': 182.793, 'train_steps_per_second': 11.436, 'total_flos': 1879617791900160.0, 'train_loss': 0.04314749269418314, 'epoch': 5.0})

In [38]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Current VRAM Allocated (MB):", torch.cuda.memory_allocated(0) / (1024**2))

CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Current VRAM Allocated (MB): 1309.25244140625


In [39]:
# Tokenize WITHOUT padding or max_length restrictions to keep RAM clean
def quick_tokenize(examples):
    return tokenizer(examples['text'], truncation=True)

# Re-map tokenized arrays cleanly
tokenized_train = train_dataset.map(quick_tokenize, batched=True, remove_columns=['text'])
tokenized_val = val_dataset.map(quick_tokenize, batched=True, remove_columns=['text'])

Map: 100%|██████████| 1419/1419 [00:00<00:00, 28492.39 examples/s]


In [42]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback
import torch

# Clear out any residual cache memory from your GPU
torch.cuda.empty_cache()

# 1. Initialize the dynamic data loader collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Re-verify the model architecture is firmly anchored on the GPU
model = model.to("cuda")

# 3. Restructure your tuned arguments block
tuned_training_args = TrainingArguments(
    output_dir="./results_distilbert_tuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.02,
    warmup_steps=manual_warmup_steps,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                        # Mixed precision handles VRAM processing
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=0          # Keeps processing single-threaded to protect RAM
)

# 4. Initialize the Trainer execution block (FIXED: removed unexpected tokenizer parameter)
tuned_trainer = Trainer(
    model=model,                      
    args=tuned_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,      # Dynamic batch loader engine handles padding now
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n--- Running Optimized Hyperparameter Fine-Tuning Loop on GPU ---")
tuned_trainer.train()


--- Running Optimized Hyperparameter Fine-Tuning Loop on GPU ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.103690,2.244862,0.687808,0.688294,0.699256,0.687808
2,0.057146,2.285691,0.694856,0.694873,0.698745,0.694856
3,0.047947,2.459725,0.704017,0.704974,0.710953,0.704017
4,0.016928,2.470335,0.706131,0.705690,0.707984,0.706131
5,0.016269,2.468827,0.708950,0.708420,0.710305,0.708950


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


TrainOutput(global_step=3550, training_loss=0.03963483561092699, metrics={'train_runtime': 229.9789, 'train_samples_per_second': 246.74, 'train_steps_per_second': 15.436, 'total_flos': 1011835084986804.0, 'train_loss': 0.03963483561092699, 'epoch': 5.0})